# Segment This Thing — Colab 데모

**Foveated Tokenization PoC**: 책상 환경 사진에 STT를 적용해 foveation이 잘 동작하는지 검증합니다.  
논문: [Segment This Thing (CVPR 2025)](https://arxiv.org/abs/2506.11131)

### 실행 순서
1. **셀 0**: 버전 호환성 체크  
2. **셀 1**: STT 설치  
3. **셀 2**: 모델 가중치 다운로드  
4. **셀 3**: 이미지 업로드  
5. **셀 4**: 모델 로드  
6. **셀 5**: 추론 + 시각화  

---
## 셀 0 — 버전 호환성 체크

STT는 **Python 3.7+**, **PyTorch 2.0+** 가 필요합니다. Colab은 기본적으로 이를 만족합니다.

In [ ]:
import sys
import torch

print("=" * 50)
print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory / 1e9:.1f} GB")
print("=" * 50)

# 최소 요구사항 검증
py_ok = sys.version_info >= (3, 7)
torch_ver = tuple(int(x) for x in torch.__version__.split(".")[:2])
torch_ok = torch_ver >= (2, 0)
cuda_ok = torch.cuda.is_available()

print(f"\n{'✅' if py_ok    else '❌'} Python 3.7+   : {'OK' if py_ok    else 'FAIL'}")
print(f"{'✅' if torch_ok else '❌'} PyTorch 2.0+  : {'OK' if torch_ok else 'FAIL'}")
print(f"{'✅' if cuda_ok  else '⚠️'} CUDA available : {'OK' if cuda_ok  else 'CPU only (느림)'}")

assert py_ok,    "Python 3.7 이상 필요"
assert torch_ok, "PyTorch 2.0 이상 필요 (scaled_dot_product_attention 사용)"
print("\n✅ 모든 호환성 체크 통과!")

---
## 셀 1 — STT 설치

In [ ]:
!pip install git+https://github.com/facebookresearch/segment_this_thing.git -q
print("✅ STT 설치 완료")

---
## 셀 2 — 모델 가중치 다운로드

| 모델 | 크기 | 용도 |
|------|------|------|
| STT-B | ~360 MB | 빠른 파이프라인 검증 |
| STT-L | ~1.2 GB | 중간 품질 |
| STT-H | ~2.5 GB | 멘토에게 보여줄 고품질 결과 |

아래에서 사용할 모델을 선택하세요 (`MODEL_SIZE` 변수).

In [ ]:
import os

# ===== 모델 선택 (b / l / h) =====
MODEL_SIZE = "b"  # 처음엔 'b'로 빠르게 검증, 결과 좋으면 'h'로 변경
# ==================================

WEIGHT_URLS = {
    "b": "https://huggingface.co/facebook/segment_this_thing/resolve/main/stt-b-qbkbmb5qsb4q2.pth",
    "l": "https://huggingface.co/facebook/segment_this_thing/resolve/main/stt-l-hrcdm1dxzwvxhd.pth",
    "h": "https://huggingface.co/facebook/segment_this_thing/resolve/main/stt-h-kj16019k5mtg3.pth",
}

os.makedirs("weights", exist_ok=True)
weight_path = f"weights/stt-{MODEL_SIZE}.pth"

if os.path.exists(weight_path):
    print(f"✅ 이미 다운로드됨: {weight_path}")
else:
    url = WEIGHT_URLS[MODEL_SIZE]
    print(f"다운로드 중: STT-{MODEL_SIZE.upper()} ...")
    !wget -q --show-progress -O {weight_path} {url}
    print(f"✅ 다운로드 완료: {weight_path}")

print(f"파일 크기: {os.path.getsize(weight_path) / 1e6:.1f} MB")

---
## 셀 3 — 이미지 업로드

책상 환경 사진을 업로드하세요. **JPG/PNG 모두 가능**합니다.  
> 💡 팁: 이미지가 1280px보다 작으면 STT의 수용 영역(1280×1280)을 벗어나 품질이 떨어질 수 있습니다. 작은 이미지는 자동으로 리사이즈됩니다.

In [ ]:
from google.colab import files
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# PIL로 로드 (uint8 RGB 보장 — matplotlib.imread는 PNG에서 float32 반환해서 STT assert 실패함)
pil_image = Image.open(image_path).convert("RGB")
H_orig, W_orig = pil_image.height, pil_image.width
print(f"원본 이미지 크기: {W_orig} x {H_orig}")

# STT 권장 최소 크기: 1280px (짧은 변 기준)
MIN_SIZE = 1280
if min(H_orig, W_orig) < MIN_SIZE:
    scale = MIN_SIZE / min(H_orig, W_orig)
    new_w = int(W_orig * scale)
    new_h = int(H_orig * scale)
    pil_image = pil_image.resize((new_w, new_h), Image.LANCZOS)
    print(f"⚠️  이미지가 작아 리사이즈: {new_w} x {new_h}")

image_np = np.array(pil_image)  # (H, W, 3), uint8
H, W = image_np.shape[:2]
print(f"사용 이미지 크기: {W} x {H}")

plt.figure(figsize=(10, 6))
plt.imshow(image_np)
plt.title("업로드된 이미지")
plt.axis("off")
plt.tight_layout()
plt.show()

---
## 셀 4 — 모델 로드

In [ ]:
import torch
import segment_this_thing
from segment_this_thing import Foveator, SegmentThisThingPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

token_size = 16
foveation_pattern = Foveator(
    token_size=token_size,
    strides=[1, 2, 4, 6, 8],
    grid_sizes=[4, 4, 6, 8, 10],
).to(device)

model_builder = getattr(segment_this_thing, f"build_segment_this_thing_{MODEL_SIZE}")
model = model_builder(
    num_tokens=foveation_pattern.get_num_tokens(),
    token_size=token_size,
)
model.load_state_dict(torch.load(weight_path, weights_only=True))
model = model.to(device).eval()

predictor = SegmentThisThingPredictor(model, foveation_pattern)
print(f"✅ STT-{MODEL_SIZE.upper()} 모델 로드 완료")

---
## 셀 5 — 추론 + 시각화

`FOV_CENTERS`에 클릭하고 싶은 위치 (x, y) 를 직접 지정하거나,  
`USE_GRID = True`로 설정하면 자동으로 격자 형태로 포인트를 생성합니다.

---
## (선택) 셀 4.5 — 좌표 확인용 헬퍼

클릭하고 싶은 위치의 픽셀 좌표를 찾기 위해 먼저 이 셀을 실행하세요.  
이미지 위에 격자와 좌표 눈금이 표시되며, 저장된 `coord_check.png`를 확대해서 원하는 버튼의 (x, y) 를 읽으면 됩니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ===== 설정 =====
GRID_STEP = 200   # 격자 간격 (픽셀). 이미지가 크면 400으로 늘리세요.
# ================

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(image_np)

# 격자선 + 좌표 눈금 표시
ax.xaxis.set_major_locator(ticker.MultipleLocator(GRID_STEP))
ax.yaxis.set_major_locator(ticker.MultipleLocator(GRID_STEP))
ax.grid(color="yellow", linewidth=0.5, alpha=0.6)
ax.tick_params(axis="both", labelsize=8, colors="red")

# 격자 교차점마다 좌표 텍스트 표시
for gx in range(0, W, GRID_STEP):
    for gy in range(0, H, GRID_STEP):
        ax.text(gx, gy, f"({gx},{gy})", fontsize=6, color="yellow",
                ha="left", va="top",
                bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.4))

ax.set_title(
    f"좌표 확인용 — 원하는 버튼 위의 노란 텍스트 (x, y) 를 읽어서\n"
    f"셀 5의 FOV_CENTERS 에 입력하세요  |  이미지 크기: {W} x {H}",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("coord_check.png", dpi=200, bbox_inches="tight")
plt.show()
print("📁 coord_check.png 저장 완료 — 파일을 다운로드해 확대하면 더 잘 보입니다.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ===== 설정 =====
USE_GRID = True          # True: 자동 격자 포인트 / False: 아래 FOV_CENTERS 직접 지정
GRID_ROWS, GRID_COLS = 2, 3  # USE_GRID=True 일 때 격자 크기
SHOW_ALL_MASKS = False    # True: 3개 마스크 전부 표시 / False: 최고 IoU 마스크만

# USE_GRID=False 일 때 사용할 포인트 (x, y) 픽셀 좌표
FOV_CENTERS = [
    (W // 4,     H // 2),
    (W // 2,     H // 2),
    (3 * W // 4, H // 2),
    (W // 2,     H // 4),
    (W // 2,     3 * H // 4),
]
# ================

if USE_GRID:
    xs = [int(W * (c + 0.5) / GRID_COLS) for c in range(GRID_COLS)]
    ys = [int(H * (r + 0.5) / GRID_ROWS) for r in range(GRID_ROWS)]
    FOV_CENTERS = [(x, y) for y in ys for x in xs]

image_tensor = torch.from_numpy(image_np)

print(f"총 {len(FOV_CENTERS)}개 포인트에서 추론 시작...")

results = []
for i, (cx, cy) in enumerate(FOV_CENTERS):
    center = torch.tensor([cx, cy], device=device)
    with torch.no_grad():
        masks, ious, foveation = predictor.get_prediction(
            image_tensor.to(device), center, return_foveation=True
        )
    results.append((cx, cy, masks, ious, foveation))
    print(f"  [{i+1}/{len(FOV_CENTERS)}] ({cx}, {cy})  IoU 최고: {ious.max().item():.3f}")

print("\n✅ 추론 완료! 시각화 중...")

# ---- 시각화 ----
# NOTE: generate_foveated_visualization은 내부적으로 CPU 텐서를 생성하므로
#       .to(device)로 명시적으로 옮겨준 뒤 색상 텐서와 연산합니다.
GREEN = torch.tensor([0x32, 0xA8, 0x52], dtype=torch.float32, device=device).view(3, 1, 1)

n = len(results)
fig, axes = plt.subplots(n, 2, figsize=(14, 5.5 * n))
if n == 1:
    axes = [axes]  # 1개일 때도 배열로 처리

for row, (cx, cy, masks, ious, foveation) in enumerate(results):
    if SHOW_ALL_MASKS:
        ious_sorted, inds = torch.sort(ious, descending=True)
        mask = masks[inds[0]]
        iou_val = ious_sorted[0].item()
    else:
        k = ious.argmax().item()
        mask, iou_val = masks[k], ious[k].item()

    # generate_foveated_visualization 반환값은 CPU → .to(device) 로 이동
    segmentation = foveation_pattern.generate_foveated_visualization(
        mask.unsqueeze(1)
    ).sigmoid().to(device)
    recon = foveation_pattern.generate_foveated_visualization(foveation).to(device)

    vis = torch.where(
        segmentation > 0.5,
        0.5 * recon.float() + 0.5 * GREEN,
        recon.float(),
    ).permute(1, 2, 0).byte().cpu().numpy()

    # 왼쪽: 원본 + 클릭 포인트
    axes[row][0].imshow(image_np)
    axes[row][0].plot(cx, cy, "r+", markersize=22, markeredgewidth=3)
    axes[row][0].set_title(f"입력 이미지  |  클릭 포인트: ({cx}, {cy})", fontsize=12)
    axes[row][0].axis("off")

    # 오른쪽: Foveated 뷰 + 분할 결과
    axes[row][1].imshow(vis)
    axes[row][1].set_title(f"Foveated Segmentation  |  IoU = {iou_val:.3f}", fontsize=12)
    axes[row][1].axis("off")

fig.suptitle(
    f"STT-{MODEL_SIZE.upper()} Foveation PoC  —  책상 환경",
    fontsize=16,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig("stt_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📁 결과 저장: stt_results.png")

---
## (선택) 셀 6 — 3개 마스크 전부 보기

특정 포인트의 3개 마스크를 IoU 순서대로 확인합니다.

In [ ]:
# 확인할 포인트 인덱스 (0부터 시작)
INSPECT_IDX = 0

cx, cy, masks, ious, foveation = results[INSPECT_IDX]
ious_sorted, inds = torch.sort(ious, descending=True)
masks_sorted = masks[inds]

# NOTE: generate_foveated_visualization 반환값은 CPU → .to(device) 로 이동
GREEN = torch.tensor([0x32, 0xA8, 0x52], dtype=torch.float32, device=device).view(3, 1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for k in range(3):
    mask = masks_sorted[k]
    iou_val = ious_sorted[k].item()

    segmentation = foveation_pattern.generate_foveated_visualization(
        mask.unsqueeze(1)
    ).sigmoid().to(device)
    recon = foveation_pattern.generate_foveated_visualization(foveation).to(device)

    vis = torch.where(
        segmentation > 0.5,
        0.5 * recon.float() + 0.5 * GREEN,
        recon.float(),
    ).permute(1, 2, 0).byte().cpu().numpy()

    axes[k].imshow(vis)
    axes[k].set_title(f"Mask {k+1}  |  IoU = {iou_val:.3f}", fontsize=12)
    axes[k].axis("off")

fig.suptitle(f"포인트 ({cx}, {cy}) — 3개 마스크 전체", fontsize=14)
plt.tight_layout()
plt.savefig(f"stt_all_masks_{INSPECT_IDX}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📁 결과 저장: stt_all_masks_{INSPECT_IDX}.png")

---
## (선택) 셀 7 — 결과 파일 다운로드

In [ ]:
from google.colab import files
import glob

for f in glob.glob("stt_*.png"):
    print(f"다운로드: {f}")
    files.download(f)